In [ ]:
import polars as pl
import pandas as pd
import joblib
import os
import gc
import os
    
# ... rest of your code ...

# --- KAGGLE CONFIGURATION ---
# 1. Input Data Path (Standard for this competition)
# Note: Check the sidebar to confirm if it's 'csv_files/test' or just 'test'
TEST_DIR = "/kaggle/input/home-credit-credit-risk-model-stability/csv_files/test"

# 2. Your Model Path (From the Dataset you created)
# REPLACE 'home-credit-baseline-model' with the exact name of your uploaded dataset folder
MODEL_DIR = "/kaggle/input/home-credit-stability-baseline-model"
MODEL_PATH = f"{MODEL_DIR}/lgbm_baseline.joblib"
CAT_LIST_PATH = f"{MODEL_DIR}/cat_cols.joblib"

def run_submission():
    print("🚀 Starting Submission Pipeline...")
    
    # 1. Load Model and Category List
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Model file not found at {MODEL_PATH}")
        print("Did you add your dataset to the notebook?")
        return
        
    model = joblib.load(MODEL_PATH)
    train_cat_cols = joblib.load(CAT_LIST_PATH)
    print("✅ Model and Metadata Loaded.")

    # 2. Load Test Data
    # Note: In the actual submission run, Kaggle swaps these files for the big hidden test set
    try:
        df_base = pl.read_csv(f"{TEST_DIR}/test_base.csv")
        df_static_0 = pl.read_csv(f"{TEST_DIR}/test_static_0_0.csv")
        df_static_1 = pl.read_csv(f"{TEST_DIR}/test_static_0_1.csv")
        
        df_static = pl.concat([df_static_0, df_static_1], how="vertical_relaxed")
        df_test = df_base.join(df_static, on="case_id", how="left")
        
        # Garbage collection to save memory on the cloud kernel
        del df_static_0, df_static_1, df_static
        gc.collect()
        
    except Exception as e:
        print(f"❌ Error loading test data: {e}")
        return

    # 3. Preprocessing
    print("🧹 Preprocessing Test Data...")
    pdf_test = df_test.to_pandas()
    
    # Build X_test using dictionary
    expected_features = model.feature_name_
    data_dict = {}
    
    print("⚙️ Aligning features...")
    for feature in expected_features:
        if feature in pdf_test.columns:
            data_dict[feature] = pdf_test[feature]
        else:
            data_dict[feature] = None
            
    X_test = pd.DataFrame(data_dict)
    
    # --- RESTORE CATEGORIES ---
    print("🔧 Restoring Categorical Types...")
    for col in train_cat_cols:
        if col in X_test.columns:
            X_test[col] = X_test[col].astype('category')

    # --- CLEAN OBJECT COLUMNS ---
    print("🧹 Cleaning remaining object columns...")
    obj_cols = X_test.select_dtypes(include=['object']).columns
    for col in obj_cols:
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

    # 5. Predict
    print("🔮 Generating Predictions...")
    scores = model.predict_proba(X_test)[:, 1]

    # 6. Save
    # Kaggle EXPECTS the file to be saved to /kaggle/working/submission.csv
    submission = pd.DataFrame({
        "case_id": df_test["case_id"],
        "score": scores
    })
    
    submission.to_csv("submission.csv", index=False)
    print("✅ submission.csv saved successfully!")
    print(submission.head())

if __name__ == "__main__":
    run_submission()